# Economic Indicators + For Loops in R
## Solution Notebook — Iterating over Macroeconomic Series

Complete answers, safer idioms (`seq_along`, pre-allocation), vectorised alternates, and a reusable simulation function.


## Flowchart

![Economic Indicators + For Loops](economic_indicators_for_loops_flowchart.png)


In [ ]:
years <- 2017:2026
inflation <- c(2.1, 2.4, 1.8, 1.2, 4.7, 8.0, 4.1, 3.4, 2.9, 2.5)
gdp_growth <- c(2.3, 2.9, 2.3, -2.8, 5.9, 2.1, 2.5, 2.0, 1.8, 2.2)
unemployment <- c(4.4, 3.9, 3.7, 8.1, 5.4, 3.6, 3.7, 4.0, 4.2, 4.1)

macro_df <- data.frame(
  year = 2022:2026,
  inflation = c(8.0, 4.1, 3.4, 2.9, 2.5),
  gdp_growth = c(1.9, 2.5, 2.1, 1.8, 2.0),
  unemployment = c(3.6, 3.7, 4.0, 4.2, 4.1),
  stringsAsFactors = FALSE
)


## 1. Basic `for` — Solutions


In [ ]:
# Task 1.1 — print "Hello" 20 times (the coding-question change)
for (i in 1:20) {
  print("Hello")
}


In [ ]:
# Task 1.2
for (i in 1:length(years)) {
  print(years[i])
}
# Safer equivalent:
for (i in seq_along(years)) {
  print(years[i])
}


## 2. Looping over vector values — Solutions


In [ ]:
# Task 2.1
for (x in inflation) {
  if (x >= 4) {
    print(paste("High inflation year:", x))
  }
}


In [ ]:
# Task 2.2
for (i in seq_along(years)) {
  print(paste0("Year ", years[i], ": inflation = ", inflation[i]))
}


## 3. Building results inside a loop — Solutions


In [ ]:
# Task 3.1 — cumulative GDP growth
cum_growth <- numeric(length(gdp_growth))
cum_growth[1] <- gdp_growth[1]
for (i in 2:length(gdp_growth)) {
  cum_growth[i] <- cum_growth[i-1] + gdp_growth[i]
}
print(data.frame(year = years, gdp_growth = gdp_growth, cum_growth = cum_growth))

# Alternate (vectorised) — preferred in production code
cum_growth_vec <- cumsum(gdp_growth)


In [ ]:
# Task 3.2 — regime labels
regime <- character(length(inflation))
for (i in seq_along(inflation)) {
  x <- inflation[i]
  if (x >= 5) {
    regime[i] <- "High"
  } else if (x >= 3) {
    regime[i] <- "Elevated"
  } else if (x >= 2) {
    regime[i] <- "Target"
  } else {
    regime[i] <- "Low"
  }
}
print(data.frame(year = years, inflation = inflation, regime = regime))

# Vectorised alternate
regime_vec <- ifelse(inflation >= 5, "High",
              ifelse(inflation >= 3, "Elevated",
              ifelse(inflation >= 2, "Target", "Low")))


## 4. Nested loops & counting — Solutions


In [ ]:
# Task 4.1 — multiplication table 1..3
for (i in 1:3) {
  for (j in 1:3) {
    print(paste(i, "x", j, "=", i*j))
  }
}


In [ ]:
# Task 4.2 — count high-inflation years
n_high <- 0
for (x in inflation) {
  if (x >= 4) {
    n_high <- n_high + 1
  }
}
print(n_high)

# Vectorised alternate
n_high_vec <- sum(inflation >= 4)


## 5. More Practice — Solutions


In [ ]:
# Practice A — running maximum
running_max <- numeric(length(inflation))
running_max[1] <- inflation[1]
for (i in 2:length(inflation)) {
  running_max[i] <- max(running_max[i-1], inflation[i])
}
print(data.frame(year = years, inflation = inflation, running_max = running_max))

# Vectorised alternate
running_max_vec <- cummax(inflation)


In [ ]:
# Practice B — years with unemployment > 5
high_unemp_years <- c()
for (i in seq_along(unemployment)) {
  if (unemployment[i] > 5) {
    high_unemp_years <- c(high_unemp_years, years[i])
  }
}
print(high_unemp_years)

# Cleaner alternate with logical indexing
high_unemp_years2 <- years[unemployment > 5]


## 6. Simulation — Solutions


In [ ]:
# Parameters
n_sims   <- 20
base_inf <- 2.5
shock_sd <- 0.8
set.seed(42)

sim_regime <- character(n_sims)
for (i in 1:n_sims) {
  shock   <- rnorm(1, mean = 0, sd = shock_sd)
  sim_inf <- base_inf + shock
  if (sim_inf >= 5) {
    sim_regime[i] <- "High"
  } else if (sim_inf >= 3) {
    sim_regime[i] <- "Elevated"
  } else if (sim_inf >= 2) {
    sim_regime[i] <- "Target"
  } else {
    sim_regime[i] <- "Low"
  }
}
print(table(sim_regime))


In [ ]:
# Function-wrapped version (recommended)
simulate_regimes <- function(n = 20, base = 2.5, sd = 0.8, seed = NULL) {
  if (!is.null(seed)) set.seed(seed)
  regimes <- character(n)
  for (i in seq_len(n)) {
    sim_inf <- base + rnorm(1, 0, sd)
    regimes[i] <- if (sim_inf >= 5) "High"
                  else if (sim_inf >= 3) "Elevated"
                  else if (sim_inf >= 2) "Target"
                  else "Low"
  }
  table(regimes)
}

print(simulate_regimes(n = 20, seed = 42))
print(simulate_regimes(n = 100, base = 3.0, sd = 1.2, seed = 123))


## Key Takeaways

1. `for (i in 1:n)` is the direct answer to “do this n times” (the original coding question).
2. Prefer `seq_along(x)` over `1:length(x)` — it behaves correctly on empty vectors.
3. Always **pre-allocate** result vectors (`numeric(n)`, `character(n)`) before filling them inside a loop.
4. Combining a `for` loop with `if / else if` is the classic way to build categorical labels or counts.
5. Many loop patterns have faster vectorised equivalents (`cumsum`, `cummax`, `sum(condition)`, logical indexing). Learn both: loops for clarity and complex logic, vectorised code for speed.
6. Wrap repeated simulation logic in a function so you can change `n_sims`, base rate or shock size without touching the loop body.
